<a href="https://colab.research.google.com/github/ParkHangah/AIFFEL_quest_eng/blob/master/Main_Quest/Quest02/%EC%B1%97%EB%B4%87_%ED%95%99%EC%8A%B5%EC%9A%A9_CSV%EB%8D%B0%EC%9D%B4%ED%84%B0_%EA%B0%80%EA%B3%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. 드라이브 연결

##### (1) 경로 설정을 사용사 환경에 맞게 해주세요

In [1]:
# 드라이브 경로 설정  by 박항아
drive_path = '#Study/Aiffel/Work' # 평상시 작업하는 드라이브 폴더 경로를 입력해 주세요.
project_name = 'GPT'       # 이번 프로젝트세 사용하는 폴더명을 입력해주세요.

##### (2) 드라이브 연결

In [ ]:
from google.colab import drive
from IPython.display import clear_output
import ipywidgets as widgets
import os

def inf(msg, style, wdth):
    inf = widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth))
    display(inf)

# 1. 구글 드라이브 마운트
print("Connecting...")
drive.mount('/content/gdrive')

# 2. 경로 설정 및 폴더 생성
base_path = os.path.join('/content/gdrive/MyDrive',drive_path)
project_path = os.path.join(base_path, project_name)

# Create the project directory if it doesn't exist
os.makedirs(project_path, exist_ok=True)

print(f"base_path: {base_path}")
# 3. 폴더 생성
print(f"Selected Google Drive root path: {base_path}")
inf('\u2714 Done','success', '50px')

### 2. 데이터를 CSV 형태로 가공

In [3]:
import os
import re
import pandas as pd
from tqdm import tqdm

In [9]:
# 1. 경로 설정 (Source [1] 기반)
DATA_PATH = os.path.join(project_path, "data")
# 처리할 폴더 리스트
data_folders = ['TS_01. KAKAO(2)']

In [10]:
# 2. 한글 정규화 및 기본 전처리 함수 (Source [2] 기반)
def clean_korean_text(text: str) -> str:
    text = str(text).strip()
    # 한글, 영문, 숫자 및 주요 문장부호 제외 제거
    text = re.sub(r"[^ㄱ-ㅎ가-힣0-9a-zA-Z\?\!\.\,\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
# 3. 폴더별 순회 및 데이터 처리
for folder_name in data_folders:
    folder_full_path = os.path.join(DATA_PATH, folder_name)

    if not os.path.exists(folder_full_path):
        print(f"경고: {folder_full_path} 경로를 찾을 수 없습니다. 건너뜁니다.")
        continue

    # [핵심] 해당 폴더 내의 모든 .txt 파일 리스트업 (비순차 파일 대응)
    file_list = [f for f in os.listdir(folder_full_path) if f.endswith('.txt')]
    print(f"[{folder_name}] 처리 중... (파일 개수: {len(file_list)})")

    current_folder_data = []

    for file_name in tqdm(file_list, desc=f"Converting {folder_name}"):
        file_path = os.path.join(folder_full_path, file_name)

        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()

            dialogues = []
            for line in lines:
                # '번호 : 텍스트' 패턴 추출 (Source [3] 데이터 구조 기반)
                match = re.match(r"^\d+\s*:\s*(.*)", line)
                if match:
                    clean_line = clean_korean_text(match.group(1))
                    if len(clean_line) >= 2: # 최소 길이 필터링
                        dialogues.append(clean_line)

            # 연속된 대화를 Q-A 쌍으로 생성
            for i in range(len(dialogues) - 1):
                current_folder_data.append({
                    'Q': dialogues[i],
                    'A': dialogues[i+1],
                    'label': 0
                })
        except Exception as e:
            print(f"파일 읽기 오류 ({file_name}): {e}")

[TS_01. KAKAO(2)] 처리 중... (파일 개수: 18000)


Converting TS_01. KAKAO(2):  23%|██▎       | 4170/18000 [57:23<3:03:01,  1.26it/s]

In [8]:
# 4. 폴더별 개별 CSV 저장
if current_folder_data:
    df = pd.DataFrame(current_folder_data)
    # 중복 제거
    df = df.drop_duplicates().dropna()

    # 저장 파일명 설정 (DATA_PATH 안에 폴더명.csv 로 저장)
    save_filename = f"{folder_name}.csv"
    save_path = os.path.join(DATA_PATH, save_filename)

    # CSV 저장 (한글 깨짐 방지를 위해 utf-8-sig 사용)
    df.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"성공: {save_filename} 저장 완료! (데이터: {len(df)}개)")
else:
    print(f"알림: {folder_name}에 처리할 데이터가 없습니다.")

print("\n모든 작업이 완료되었습니다.")

성공: TS_01. KAKAO(1).csv 저장 완료! (데이터: 381981개)

모든 작업이 완료되었습니다.
